* XGBoost ?? ??

In [ ]:
import xgboost

print(xgboost.__version__)

### ??? ?? XGBoost ?? ? ???? ??? ??

In [ ]:
import xgboost as xgb
from xgboost import plot_importance
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

dataset = load_breast_cancer()
X_features= dataset.data
y_label = dataset.target

cancer_df = pd.DataFrame(data=X_features, columns=dataset.feature_names)
cancer_df['target']= y_label
cancer_df.head(3)


In [ ]:
print(dataset.target_names)
print(cancer_df['target'].value_counts())

In [ ]:
# cancer_df?? feature? DataFrame? Label? Series ?? ??
# ? ??? ??? Label?. Feature? DataFrame? cancer_df? ??? ???? ? ??? ??? ????? :-1 ?????? ??.
X_features = cancer_df.iloc[:, :-1]
y_label = cancer_df.iloc[:, -1]

# ?? ??? ? 80%? ??? ???, 20%? ???? ??? ??
X_train, X_test, y_train, y_test=train_test_split(X_features, y_label,
                                         test_size=0.2, random_state=156 )

# ??? ?? X_train, y_train? ?? ??? 90%? ??? 10%? ??? ???? ??
X_tr, X_val, y_tr, y_val= train_test_split(X_train, y_train, test_size=0.1, random_state=156 )
print(X_train.shape , X_test.shape)
print(X_tr.shape, X_val.shape)

In [ ]:
# ?? ??? XGBoost?? DataFrame?? DMatrix ??? ?? ?? X_train.values? ??? ??.
# ??, ??, ???? DMatrix? ??.
dtr = xgb.DMatrix(data=X_tr, label=y_tr)
dval = xgb.DMatrix(data=X_val, label=y_val)
dtest = xgb.DMatrix(data=X_test , label=y_test)

In [ ]:
params = { 'max_depth':3,
          'eta': 0.05,
          'objective':'binary:logistic',
          'eval_metric':'logloss'
         }
num_rounds = 400

In [ ]:
# ?? ??? ?? 'train' ?? ?? ??? ?? 'eval' ? ?????. 
eval_list = [(dtr,'train'),(dval,'eval')] # ?? eval_list = [(dval,'eval')] ? ???? ??. 

# ??? ????? early stopping ????? train( ) ??? ????? ??
xgb_model = xgb.train(params = params , dtrain=dtr , num_boost_round=num_rounds , \
                      early_stopping_rounds=50, evals=eval_list )

In [ ]:
pred_probs = xgb_model.predict(dtest)
print('predict( ) ?? ???? 10?? ??, ?? ?? ??? ???')
print(np.round(pred_probs[:10],3))

# ?? ??? 0.5 ?? ?? 1 , ??? ??? 0 ?? ??? ???? List ??? preds? ?? 
preds = [ 1 if x > 0.5 else 0 for x in pred_probs ]
print('??? 10?? ??:',preds[:10])

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score

def get_clf_eval(y_test, pred=None, pred_proba=None):
    confusion = confusion_matrix( y_test, pred)
    accuracy = accuracy_score(y_test , pred)
    precision = precision_score(y_test , pred)
    recall = recall_score(y_test , pred)
    f1 = f1_score(y_test,pred)
    # ROC-AUC ?? 
    roc_auc = roc_auc_score(y_test, pred_proba)
    print('?? ??')
    print(confusion)
    # ROC-AUC print ??
    print('???: {0:.4f}, ???: {1:.4f}, ???: {2:.4f},\
    F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))

In [ ]:
get_clf_eval(y_test , preds, pred_probs)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

fig, ax = plt.subplots(figsize=(10, 12))
plot_importance(xgb_model, ax=ax)
plt.savefig('p239_xgb_feature_importance.tif', format='tif', dpi=300, bbox_inches='tight')

### ???? ?? XGBoost? ?? ? ??

In [ ]:
# ???? ?? XGBoost ???? XGBClassifier ???
from xgboost import XGBClassifier

xgb_wrapper = XGBClassifier(n_estimators=400, learning_rate=0.1, max_depth=3)
xgb_wrapper.fit(X_train, y_train)
w_preds = xgb_wrapper.predict(X_test)
w_pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]

In [ ]:
get_clf_eval(y_test , w_preds, w_pred_proba)

In [ ]:
from xgboost import XGBClassifier

xgb_wrapper = XGBClassifier(n_estimators=400, learning_rate=0.1, max_depth=3)
evals = [(X_test, y_test)]
xgb_wrapper.fit(X_train, y_train, early_stopping_rounds=100, eval_metric="logloss", 
                eval_set=evals, verbose=True)

ws100_preds = xgb_wrapper.predict(X_test)
ws100_pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]

In [ ]:
get_clf_eval(y_test , ws100_preds, ws100_pred_proba)

In [ ]:
# early_stopping_rounds? 10?? ???? ? ??. 
xgb_wrapper.fit(X_train, y_train, early_stopping_rounds=10, 
                eval_metric="logloss", eval_set=evals,verbose=True)

ws10_preds = xgb_wrapper.predict(X_test)
ws10_pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]
get_clf_eval(y_test , ws10_preds, ws10_pred_proba)

In [ ]:
from xgboost import plot_importance
import matplotlib.pyplot as plt
%matplotlib inline

fig, ax = plt.subplots(figsize=(10, 12))
# ???? ?? ???? ???? ??. 
plot_importance(xgb_wrapper, ax=ax)
